# 🧪 Aula 14 — Aproximação normal num teste A/B e revisão dos modelos

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de um aplicativo de compras**. Na primeira parte, o time analisa um **teste A/B** (testar uma versão nova de uma tela com um grupo de usuários) e usa a ferramenta da Aula 14: a **aproximação da binomial pela normal**. Na segunda parte, revisamos **todos os modelos** do bloco com situações do dia a dia de TI.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats`**: o "pacote de tabelas" do Python. Para cada modelo do curso há uma tabela pronta (`binom`, `norm`, `poisson`, `geom`, `hypergeom`, `expon`, `uniform`), e todas funcionam do mesmo jeito:
  - `.pmf(k, ...)` dá $P(X = k)$ (modelos **discretos**);
  - `.cdf(x, ...)` dá a acumulada $P(X \le x)$ (todos os modelos).

Criamos também o **sorteador** (`rng`) com a **semente 42**.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula (ou a tabela) diz.

In [ ]:
import numpy as np                                     # calculadora de números e sorteios
import matplotlib.pyplot as plt                        # gráficos
from scipy.stats import binom, norm, poisson, geom, hypergeom, expon, uniform   # as "tabelas"

rng = np.random.default_rng(42)                        # sorteador, semente 42
print("Tudo pronto!")

## 1. O teste A/B do botão de compra

### A situação
Hoje, **10%** dos visitantes da página de um produto clicam em "Comprar". O time de design criou um botão novo e mostrou a versão nova para **400 visitantes**. Resultado: **50 compras**.

### A pergunta
Se o botão novo **não mudasse nada** (a taxa continuasse 10%), qual seria a chance de aparecerem **50 ou mais** compras em 400 visitantes, só por acaso?

### A ferramenta da aula
Seja $X$ = número de compras entre os 400 visitantes. Cada visitante compra ou não, de forma independente, com $p = 0{,}10$: $X \sim B(400;\ 0{,}10)$. A conta exata exigiria somar 351 termos binomiais ($k = 50, 51, \dots, 400$). Como $n$ é grande, usamos a **normal** com a mesma média e variância:
$$\mu = np = 40, \qquad \sigma^2 = np(1-p) = 36, \qquad \sigma = 6 .$$

E, como a binomial é discreta e a normal é contínua, fazemos a **correção de continuidade**: "50 ou mais" vira "a partir de 49,5".

### A resposta pela conta
$$P(X \ge 50) \approx P(Y \ge 49{,}5) = P\left(Z \ge \frac{49{,}5 - 40}{6}\right) = P(Z \ge 1{,}58) \approx 0{,}0567 .$$

### O que o código faz
Calcula a mesma probabilidade de três jeitos, para comparar:
1. **binomial exata** (o computador soma os 351 termos sem reclamar);
2. **normal sem** correção de continuidade (a partir de 50);
3. **normal com** correção de continuidade (a partir de 49,5).

E ainda **simula 100.000 testes A/B** em que o botão não muda nada, para ver em quantos aparecem 50 ou mais compras.

In [ ]:
n, p = 400, 0.10
mu = n * p                                             # 40
sigma = np.sqrt(n * p * (1 - p))                       # 6

exata = 1 - binom.cdf(49, n, p)                        # P(X >= 50) = 1 - P(X <= 49)
sem_correcao = 1 - norm.cdf((50 - mu) / sigma)
com_correcao = 1 - norm.cdf((49.5 - mu) / sigma)

testes = rng.binomial(n, p, size=100_000)              # 100.000 testes A/B "sem efeito"

print(f"Binomial exata:               {exata:.4f}")
print(f"Normal sem correção (z = {(50 - mu) / sigma:.2f}): {sem_correcao:.4f}")
print(f"Normal com correção (z = {(49.5 - mu) / sigma:.2f}): {com_correcao:.4f}")
print(f"Simulação:                    {(testes >= 50).mean():.4f}")

**Como ler:**
- A conta exata dá cerca de **6,0%**. A normal **com correção** chega mais perto (5,7%) do que a normal **sem correção** (4,8%): o meio ponto faz diferença.
- Ou seja, se o botão não mudasse nada, ver 50 ou mais compras aconteceria em cerca de **6 de cada 100** testes. Não é impossível, mas é pouco comum.

**Para pensar:** o resultado de 50 compras é um **indício** de que o botão novo funciona, mas não uma prova — 6% de chance de acontecer por acaso não é desprezível. Decidir "quando um resultado é raro o bastante" é exatamente o assunto dos **testes de hipóteses**, no fim do curso.

## 2. Por que a aproximação funciona?

### A pergunta
A aula comparou a binomial com $n = 4$ e $p = 0{,}5$ com a normal de mesma média e variância. **O que acontece quando $n$ cresce?**

### O que o código faz
Desenha, lado a lado:
- à esquerda, a binomial $B(4;\ 0{,}5)$ em barras, com a curva da normal $N(2;\ 1)$ por cima (o exemplo da aula);
- à direita, a binomial do teste A/B, $B(400;\ 0{,}10)$, com a curva da normal $N(40;\ 36)$.

E imprime, para $n = 4$, a probabilidade de cada valor pela binomial e pela normal com correção de continuidade ($P(X = k) \approx P(k - 0{,}5 < Y < k + 0{,}5)$).

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

for eixo, (n_, p_) in zip(eixos, [(4, 0.5), (400, 0.10)]):
    m, s = n_ * p_, np.sqrt(n_ * p_ * (1 - p_))
    k = np.arange(max(0, int(m - 4 * s)), int(m + 4 * s) + 1)     # valores perto da média
    x = np.linspace(k.min() - 1, k.max() + 1, 400)
    eixo.bar(k, binom.pmf(k, n_, p_), color="lightsteelblue", edgecolor="black", label="binomial")
    eixo.plot(x, norm.pdf(x, m, s), color="firebrick", linewidth=2, label="normal")
    eixo.set_title(f"B({n_}; {p_}) e N({m:.0f}; {s**2:.0f})")
    eixo.legend()
plt.show()

print(" k   binomial   normal (com correção)")
for k in range(5):
    aprox = norm.cdf(k + 0.5, 2, 1) - norm.cdf(k - 0.5, 2, 1)
    print(f" {k}   {binom.pmf(k, 4, 0.5):.4f}     {aprox:.4f}")

**Como ler:**
- Com $n = 4$, a normal já acompanha o "formato" das barras, mas os números diferem um pouco (0,3750 contra 0,3829 em $k = 2$).
- Com $n = 400$, a curva passa praticamente pelo topo de todas as barras: a aproximação é excelente.

A regra prática da aula: a aproximação é boa quando $np$ e $n(1-p)$ são **ambos grandes** (é comum pedir pelo menos 5). No teste A/B, $np = 40$ e $n(1-p) = 360$.

## 3. Revisão: qual modelo usar?

Esta seção revisa os modelos do bloco. Para cada situação, **tente responder antes de olhar a tabela**: qual o modelo e quais os parâmetros?

| | Situação | Pergunta |
|---|---|---|
| a | Um teste automatizado falha com probabilidade 0,02 a cada execução, de forma independente. Ele roda até falhar pela primeira vez. | Chance de rodar **mais de 50 vezes** sem falhar? |
| b | De 80 servidores, 12 têm uma vulnerabilidade. Um auditor sorteia 10 servidores diferentes. | Chance de encontrar **pelo menos 1** vulnerável? |
| c | Cada um dos 30 *builds* (compilações) do dia falha com probabilidade 0,05, de forma independente. | Chance de **nenhum** build falhar? |
| d | Um site registra, em média, 2 erros "página não encontrada" por minuto. | Chance de **5 ou mais** erros em um minuto? |
| e | Ainda no site do item d. | Chance de o próximo erro chegar em **menos de 10 segundos**? |
| f | Um serviço reinicia num minuto sorteado ao acaso entre 0 e 30. | Chance de reiniciar nos **primeiros 5 minutos**? |
| g | O tempo de resposta de uma API é normal com média 200 ms e desvio padrão 30 ms. | Chance de uma resposta passar de **250 ms**? |

### As respostas
| | Modelo | Por quê | Conta |
|---|---|---|---|
| a | Geométrica, $p = 0{,}02$ | repetições até o 1º sucesso (a "falha" do teste) | $P(X > 50) = 0{,}98^{50}$ |
| b | Hipergeométrica, $N = 80$, $r = 12$, $n = 10$ | sorteio **sem reposição** de população finita | $1 - P(X = 0)$ |
| c | Binomial, $n = 30$, $p = 0{,}05$ | número fixo de ensaios independentes | $P(X = 0) = 0{,}95^{30}$ |
| d | Poisson, $\lambda = 2$ | contagem num intervalo de tempo | $1 - P(X \le 4)$ |
| e | Exponencial, $\lambda = 2$ por minuto | **tempo** entre ocorrências de uma Poisson | $P(T < 1/6) = 1 - e^{-2/6}$ |
| f | Uniforme $U[0, 30]$ | todos os minutos igualmente prováveis | $5/30$ |
| g | Normal $N(200;\ 30^2)$ | dado no enunciado | $1 - \Phi\big(\frac{250 - 200}{30}\big)$ |

### O que o código faz
Calcula a probabilidade de cada item com a "tabela" do modelo certo. Repare que todas as linhas têm o mesmo formato: **modelo**`.cdf` ou **modelo**`.pmf`, com os parâmetros da situação.

In [ ]:
respostas = {
    "a) geométrica":      1 - geom.cdf(50, 0.02),                       # P(X > 50)
    "b) hipergeométrica": 1 - hypergeom.pmf(0, 80, 12, 10),             # 1 - P(X = 0)
    "c) binomial":        binom.pmf(0, 30, 0.05),                       # P(X = 0)
    "d) Poisson":         1 - poisson.cdf(4, 2),                        # P(X >= 5)
    "e) exponencial":     expon.cdf(1 / 6, scale=1 / 2),                # P(T < 10 s); scale = média = 1/λ
    "f) uniforme":        uniform.cdf(5, loc=0, scale=30),              # P(X < 5) em U[0, 30]
    "g) normal":          1 - norm.cdf(250, loc=200, scale=30),         # P(X > 250)
}
for item, prob in respostas.items():
    print(f"{item:20s} {prob:.4f}")

**Como ler:**
- a) cerca de **36%** de chance de o teste rodar mais de 50 vezes sem falhar;
- b) cerca de **83%** de chance de o auditor encontrar pelo menos um servidor vulnerável;
- c) só cerca de **21%** de chance de um dia sem nenhum build quebrado;
- d) cerca de **5%** dos minutos têm 5 ou mais erros;
- e) cerca de **28%** de chance de o próximo erro chegar em menos de 10 segundos;
- f) $5/30 \approx 17\%$;
- g) cerca de **4,8%** das respostas passam de 250 ms.

**Dica para a prova:** o que decide o modelo é **o que a variável mede**. Contagem de sucessos em $n$ ensaios independentes → binomial; sem reposição → hipergeométrica; tentativas até o 1º sucesso → geométrica; contagem num intervalo → Poisson; **tempo** até um evento → exponencial; tudo igualmente provável num intervalo → uniforme; forma de sino → normal.

## Para fechar

Neste notebook:

- a **aproximação normal da binomial** ($\mu = np$, $\sigma^2 = np(1-p)$, com correção de continuidade) analisou um teste A/B com 400 visitantes;
- os gráficos mostraram por que a aproximação melhora quando $n$ cresce;
- a revisão percorreu os **sete modelos** do bloco, com o mesmo formato de conta para todos.

Na **Aula 15** começa a **inferência estatística**: em vez de supor que conhecemos $p$ ou $\mu$, vamos **estimá-los a partir de uma amostra**.